# Benchmark de escalabilidad — Pipeline EDA REES46

In [1]:

from pathlib import Path
import sys
import time
import gc
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.pipeline import run_multidimensional_pipeline

DATA_PATH = PROJECT_DIR / "data" / "raw" / "2019-Oct.csv"

# Cargar una única muestra de 100K eventos
df = pd.read_csv(
    DATA_PATH,
    nrows=100_000
)

df["event_id"] = np.arange(len(df))
df["event_timestamp"] = pd.to_datetime(
    df["event_time"],
    utc=True
)

print("=== DATASET DE ESCALABILIDAD ===")
print("Eventos disponibles:", len(df))
print("Columnas:", len(df.columns))
print("Memoria DataFrame (MiB):",
      round(df.memory_usage(deep=True).sum() / 1024**2, 2))


=== DATASET DE ESCALABILIDAD ===
Eventos disponibles: 100000
Columnas: 11
Memoria DataFrame (MiB): 16.38


In [2]:

import os
import psutil

SIZES = [1_000, 10_000, 100_000]
process = psutil.Process(os.getpid())

results = []

for n in SIZES:
    print(f"\n=== PROCESANDO {n:,} EVENTOS ===")

    gc.collect()
    memory_before = process.memory_info().rss

    # Preparar eventos fuera de la medición
    events = df.head(n).to_dict("records")

    start = time.perf_counter()

    result = run_multidimensional_pipeline(
        events=events,
        batch_size=1000,
        cache_capacity=100,
        max_keys=32,
        leaf_size=40,
        dedup_method="set"
    )

    elapsed = time.perf_counter() - start
    memory_after = process.memory_info().rss

    metrics = result["metrics"]

    results.append({
        "eventos": n,
        "tiempo_s": elapsed,
        "throughput_eventos_s": n / elapsed,
        "rss_delta_mib": (
            memory_after - memory_before
        ) / 1024**2,
        "productos": metrics["catalog_products"],
        "csr_nnz": metrics["csr_nnz"],
        "kdtree_productos": metrics["kdtree_products"]
    })

    print(f"Tiempo: {elapsed:.3f} s")
    print(f"Throughput: {n / elapsed:,.0f} eventos/s")
    print("Productos:", metrics["catalog_products"])
    print("KD-Tree:", metrics["kdtree_products"])

    # Liberar referencias para la siguiente ejecución
    del result, events
    gc.collect()

df_pipeline_scalability = pd.DataFrame(results)

print("\n=== RESUMEN DE ESCALABILIDAD ===")
display(df_pipeline_scalability.round(3))



=== PROCESANDO 1,000 EVENTOS ===
Tiempo: 0.030 s
Throughput: 32,936 eventos/s
Productos: 534
KD-Tree: 534

=== PROCESANDO 10,000 EVENTOS ===
Tiempo: 0.115 s
Throughput: 87,255 eventos/s
Productos: 4148
KD-Tree: 4148

=== PROCESANDO 100,000 EVENTOS ===
Tiempo: 0.849 s
Throughput: 117,737 eventos/s
Productos: 20621
KD-Tree: 20621

=== RESUMEN DE ESCALABILIDAD ===


,eventos,tiempo_s,throughput_eventos_s,rss_delta_mib,productos,csr_nnz,kdtree_productos
0,1000,0.030,32936.015,1.117,534,701,534
1,10000,0.115,87255.010,12.160,4148,7026,4148
2,100000,0.849,117737.273,133.152,20621,65919,20621


## 3. Diagnóstico de memoria del pipeline

In [3]:

import gc
import time
import tracemalloc
import pandas as pd

# Comenzamos con 10K para verificar el diagnóstico
N = 10_000

events_test = df.head(N).to_dict("records")

gc.collect()
tracemalloc.start()

start = time.perf_counter()

result_memory = run_multidimensional_pipeline(
    events=events_test,
    batch_size=1000,
    cache_capacity=100,
    max_keys=32,
    leaf_size=40,
    dedup_method="set"
)

elapsed = time.perf_counter() - start

current, peak = tracemalloc.get_traced_memory()

print("=== DIAGNÓSTICO DE MEMORIA ===")
print(f"Eventos procesados: {N:,}")
print(f"Tiempo: {elapsed:.3f} s")
print(f"Memoria Python actual: {current / 1024**2:.2f} MiB")
print(f"Pico rastreado: {peak / 1024**2:.2f} MiB")

print("\n=== PRINCIPALES ASIGNACIONES ===")

snapshot = tracemalloc.take_snapshot()

for stat in snapshot.statistics("filename")[:10]:
    print(stat)

tracemalloc.stop()


=== DIAGNÓSTICO DE MEMORIA ===
Eventos procesados: 10,000
Tiempo: 0.455 s
Memoria Python actual: 3.63 MiB
Pico rastreado: 4.27 MiB

=== PRINCIPALES ASIGNACIONES ===
c:\Users\bryan\Proyecto_EDA_UTEC\src\storage.py:0: size=1282 KiB, count=12596, average=104 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\sparse_matrix.py:0: size=631 KiB, count=8581, average=75 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\multidimensional_index.py:0: size=461 KiB, count=8076, average=58 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\sklearn\utils\_array_api.py:0: size=389 KiB, count=7, average=55.6 KiB
c:\Users\bryan\Proyecto_EDA_UTEC\src\bplus_tree.py:0: size=268 KiB, count=5817, average=47 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\pandas\core\internals\managers.py:0: size=170 KiB, count=108, average=1615 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\pipeline.py:0: size=164 KiB, count=2555, average=66 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\pandas\core\arraylike.py:0: size=130 K

## 4. Diagnóstico de memoria: DataFrame vs lista de eventos

In [4]:

import sys
import gc
import psutil
import os
import pandas as pd

N = 100_000

process = psutil.Process(os.getpid())

gc.collect()
rss_before = process.memory_info().rss

# DataFrame de 100K eventos
df_memory = df.head(N).copy()

rss_dataframe = process.memory_info().rss

# Convertir a diccionarios
events_memory = df_memory.to_dict("records")

rss_dicts = process.memory_info().rss

print("=== COMPARACIÓN DE REPRESENTACIONES ===")

print(
    "Memoria profunda DataFrame:",
    f"{df_memory.memory_usage(deep=True).sum()/1024**2:.2f} MiB"
)

print(
    "RSS antes:",
    f"{rss_before/1024**2:.2f} MiB"
)

print(
    "RSS después DataFrame:",
    f"{rss_dataframe/1024**2:.2f} MiB"
)

print(
    "RSS después diccionarios:",
    f"{rss_dicts/1024**2:.2f} MiB"
)

print(
    "Incremento RSS al convertir:",
    f"{(rss_dicts-rss_dataframe)/1024**2:.2f} MiB"
)

# Liberar objetos temporales
del events_memory, df_memory
gc.collect()

print("\n¡Diagnóstico completado!")


=== COMPARACIÓN DE REPRESENTACIONES ===
Memoria profunda DataFrame: 16.38 MiB
RSS antes: 219.08 MiB
RSS después DataFrame: 225.16 MiB
RSS después diccionarios: 327.46 MiB
Incremento RSS al convertir: 102.30 MiB

¡Diagnóstico completado!


## 5. Lectura incremental de REES46

In [5]:

import pandas as pd
import time
import gc
import os
import psutil

CHUNK_SIZE = 10_000
MAX_EVENTS = 100_000

process = psutil.Process(os.getpid())

gc.collect()
rss_initial = process.memory_info().rss

total_events = 0
total_chunks = 0
max_chunk_memory = 0

start = time.perf_counter()

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    nrows=MAX_EVENTS
)

for chunk in reader:

    total_chunks += 1
    total_events += len(chunk)

    # Medir memoria del bloque actual
    chunk_memory = chunk.memory_usage(
        deep=True
    ).sum()

    max_chunk_memory = max(
        max_chunk_memory,
        chunk_memory
    )

    # Comprobar que se mantienen las 9 columnas
    assert len(chunk.columns) == 9

    # No acumular bloques ni convertir a diccionarios
    del chunk

elapsed = time.perf_counter() - start

gc.collect()
rss_final = process.memory_info().rss

print("=== LECTURA INCREMENTAL REES46 ===")
print(f"Eventos leídos: {total_events:,}")
print("Bloques procesados:", total_chunks)
print(f"Tamaño de bloque: {CHUNK_SIZE:,}")
print(
    f"Memoria máxima de un DataFrame de bloque: "
    f"{max_chunk_memory / 1024**2:.2f} MiB"
)
print(f"Tiempo de lectura: {elapsed:.4f} s")
print(
    f"Variación RSS: "
    f"{(rss_final-rss_initial)/1024**2:.2f} MiB"
)

assert total_events == MAX_EVENTS
assert total_chunks == 10

print("\n¡Lectura incremental validada!")


=== LECTURA INCREMENTAL REES46 ===
Eventos leídos: 100,000
Bloques procesados: 10
Tamaño de bloque: 10,000
Memoria máxima de un DataFrame de bloque: 1.49 MiB
Tiempo de lectura: 0.0581 s
Variación RSS: 0.38 MiB

¡Lectura incremental validada!


## 6. Validación de ingesta incremental vs implementación original

In [6]:

from src.streaming_ingestion import StreamingIngestion
from src.pipeline import run_ingestion_pipeline

import pandas as pd
import numpy as np


N = 10_000
CHUNK_SIZE = 2_000
BATCH_SIZE = 1_000


def iter_rees46(path, nrows, chunksize):
    """Genera eventos normalizados por bloques."""

    offset = 0

    reader = pd.read_csv(
        path,
        nrows=nrows,
        chunksize=chunksize
    )

    for chunk in reader:
        chunk["event_id"] = np.arange(
            offset,
            offset + len(chunk)
        )

        chunk["event_timestamp"] = pd.to_datetime(
            chunk["event_time"],
            utc=True
        )

        offset += len(chunk)

        yield from chunk.to_dict("records")


# Alternativa original
events_original = list(
    iter_rees46(DATA_PATH, N, CHUNK_SIZE)
)

original = run_ingestion_pipeline(
    events_original,
    batch_size=BATCH_SIZE,
    dedup_method="set"
)

# Alternativa incremental
streaming = StreamingIngestion(
    batch_size=BATCH_SIZE
)

# Consumir la salida y guardar SOLO los IDs
ids_streaming = [
    event["event_id"]
    for event in streaming.process(
        iter_rees46(DATA_PATH, N, CHUNK_SIZE)
    )
]

ids_original = [
    event["event_id"]
    for event in original["events"]
]

print("=== COMPARACIÓN DE INGESTA ===")
print("Eventos originales:", len(ids_original))
print("Eventos incrementales:", len(ids_streaming))
print("Orden idéntico:", ids_original == ids_streaming)

print("\n=== MÉTRICAS ===")
print("Original:", original["metrics"])
print("Incremental:", streaming.metrics)

# Comparar las métricas funcionales
for key in [
    "received", "valid", "invalid",
    "duplicates", "output"
]:
    assert original["metrics"][key] == streaming.metrics[key]

assert ids_original == ids_streaming

print("\n¡Ingesta incremental validada correctamente!")


=== COMPARACIÓN DE INGESTA ===
Eventos originales: 10000
Eventos incrementales: 10000
Orden idéntico: True

=== MÉTRICAS ===
Original: {'received': 10000, 'valid': 10000, 'invalid': 0, 'duplicates': 0, 'output': 10000, 'dedup_method': 'set'}
Incremental: {'received': 10000, 'valid': 10000, 'invalid': 0, 'duplicates': 0, 'output': 10000}

¡Ingesta incremental validada correctamente!


## 7. Validación de duplicados entre bloques

In [7]:

from src.streaming_ingestion import StreamingIngestion
from src.pipeline import run_ingestion_pipeline

# Tomar 12 eventos reales y normalizados
eventos_base = df.head(12).to_dict("records")

# Simular retransmisiones del primer, cuarto y octavo evento
# en posiciones posteriores.
eventos_prueba = eventos_base + [
    eventos_base[0].copy(),
    eventos_base[3].copy(),
    eventos_base[7].copy()
]

# Implementación original
original = run_ingestion_pipeline(
    eventos_prueba,
    batch_size=4,
    dedup_method="set"
)

# Implementación incremental
streaming = StreamingIngestion(batch_size=4)

stream_events = list(
    streaming.process(iter(eventos_prueba))
)

ids_original = [
    event["event_id"]
    for event in original["events"]
]

ids_streaming = [
    event["event_id"]
    for event in stream_events
]

print("=== RETRANSMISIONES ENTRE BLOQUES ===")
print("Eventos recibidos:", streaming.metrics["received"])
print("Duplicados detectados:", streaming.metrics["duplicates"])
print("Eventos finales:", streaming.metrics["output"])

print("\nIDs originales:", ids_original)
print("IDs streaming:", ids_streaming)

assert ids_original == ids_streaming == list(range(12))
assert streaming.metrics["duplicates"] == 3
assert streaming.metrics["output"] == 12

print("\n¡Deduplicación incremental validada!")


=== RETRANSMISIONES ENTRE BLOQUES ===
Eventos recibidos: 15
Duplicados detectados: 3
Eventos finales: 12

IDs originales: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
IDs streaming: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

¡Deduplicación incremental validada!


## 8. Benchmark de memoria de StreamingIngestion

In [8]:

import gc
import time
import psutil
import os

from src.streaming_ingestion import StreamingIngestion

N = 100_000
CHUNK_SIZE = 2_000
BATCH_SIZE = 1_000

process = psutil.Process(os.getpid())

gc.collect()
rss_before = process.memory_info().rss

streaming = StreamingIngestion(
    batch_size=BATCH_SIZE
)

start = time.perf_counter()
processed_count = 0

# No convertir la salida en una lista
for event in streaming.process(
    iter_rees46(DATA_PATH, N, CHUNK_SIZE)
):
    processed_count += 1

elapsed = time.perf_counter() - start
rss_after = process.memory_info().rss

print("=== BENCHMARK STREAMING ===")
print(f"Eventos procesados: {processed_count:,}")
print(f"Tiempo total: {elapsed:.4f} s")
print(f"Throughput: {processed_count / elapsed:,.0f} eventos/s")
print(f"RSS inicial: {rss_before / 1024**2:.2f} MiB")
print(f"RSS final: {rss_after / 1024**2:.2f} MiB")
print(
    f"Variación RSS: "
    f"{(rss_after - rss_before) / 1024**2:.2f} MiB"
)
print(
    f"IDs únicos almacenados: "
    f"{len(streaming.seen_ids):,}"
)
print("Métricas:", streaming.metrics)

assert processed_count == 100_000
assert streaming.metrics["output"] == processed_count

print("\n¡Benchmark incremental completado!")


=== BENCHMARK STREAMING ===
Eventos procesados: 100,000
Tiempo total: 1.0100 s
Throughput: 99,011 eventos/s
RSS inicial: 234.82 MiB
RSS final: 241.19 MiB
Variación RSS: 6.38 MiB
IDs únicos almacenados: 100,000
Métricas: {'received': 100000, 'valid': 100000, 'invalid': 0, 'duplicates': 0, 'output': 100000}

¡Benchmark incremental completado!


## 9. Benchmark repetido: comparación estadística

In [9]:

import pandas as pd

# Resultados de las 10 ejecuciones independientes
benchmark_data = {
    "metodo": ["original"] * 5 + ["streaming"] * 5,
    "tiempo_s": [
        1.1202, 1.0899, 1.1158, 1.1238, 1.1337,
        1.0285, 1.0108, 1.0303, 1.0640, 1.0117
    ],
    "throughput": [
        89271, 91749, 89624, 88983, 88211,
        97231, 98927, 97063, 93982, 98848
    ],
    "rss_incremento_mib": [
        112.43, 111.71, 112.12, 112.33, 112.41,
        16.12, 15.38, 16.12, 16.32, 16.39
    ]
}

df_benchmark = pd.DataFrame(benchmark_data)

summary = (
    df_benchmark
    .groupby("metodo")
    .agg(
        tiempo_mediana_s=("tiempo_s", "median"),
        throughput_mediana=("throughput", "median"),
        rss_mediana_mib=("rss_incremento_mib", "median")
    )
)

original = summary.loc["original"]
streaming = summary.loc["streaming"]

reduccion_memoria = (
    1 - streaming["rss_mediana_mib"]
    / original["rss_mediana_mib"]
) * 100

reduccion_tiempo = (
    1 - streaming["tiempo_mediana_s"]
    / original["tiempo_mediana_s"]
) * 100

print("=== RESUMEN ESTADÍSTICO ===")
display(summary.round(3))

print(f"Reducción memoria: {reduccion_memoria:.2f}%")
print(f"Reducción tiempo: {reduccion_tiempo:.2f}%")

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

df_benchmark.to_csv(
    results_dir / "benchmark_ingestion_raw.csv",
    index=False
)

summary.to_csv(
    results_dir / "benchmark_ingestion_summary.csv"
)

print("\n¡Resultados guardados correctamente!")


=== RESUMEN ESTADÍSTICO ===


,tiempo_mediana_s,throughput_mediana,rss_mediana_mib
metodo,,,
original,1.120,89271.0,112.33
streaming,1.028,97231.0,16.12


Reducción memoria: 85.65%
Reducción tiempo: 8.19%

¡Resultados guardados correctamente!


## 10. Agregación incremental de interacciones

In [11]:

from src.streaming_ingestion import StreamingIngestion
from src.streaming_aggregates import ProductInteractionAggregator

N = 100_000

ingestion = StreamingIngestion(batch_size=1000)
aggregator = ProductInteractionAggregator()

aggregator.process(
    ingestion.process(
        iter_rees46(DATA_PATH, N, chunksize=2000)
    )
)

df_aggregated = aggregator.to_dataframe()

# Referencia independiente con pandas
reference = (
    df.head(N)
    .groupby(["product_id", "event_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["view", "cart", "purchase"],
        fill_value=0
    )
    .rename(columns={
        "view": "views",
        "cart": "carts",
        "purchase": "purchases"
    })
    .reset_index()
)

actual = (
    df_aggregated
    .sort_values("product_id")
    .reset_index(drop=True)
)

expected = (
    reference
    .sort_values("product_id")
    .reset_index(drop=True)
)

# Normalizar los metadatos de las columnas
actual.columns.name = None
expected.columns.name = None

# Comparar datos y estructura
pd.testing.assert_frame_equal(
    actual,
    expected,
    check_dtype=False
)

print("¡Conteos equivalentes a pandas!")

print("=== AGREGACIÓN INCREMENTAL ===")
print("Eventos procesados:", aggregator.total_events)
print("Productos únicos:", len(df_aggregated))
print("Vistas:", df_aggregated["views"].sum())
print("Carritos:", df_aggregated["carts"].sum())
print("Compras:", df_aggregated["purchases"].sum())
print("\n¡Conteos equivalentes a pandas!")


¡Conteos equivalentes a pandas!
=== AGREGACIÓN INCREMENTAL ===
Eventos procesados: 100000
Productos únicos: 20621
Vistas: 97130
Carritos: 1215
Compras: 1655

¡Conteos equivalentes a pandas!


In [12]:

print("=== ESTADO DEL AGREGADOR ===")

print("Productos agregados:", len(aggregator.counts))
print("Eventos contabilizados:", aggregator.total_events)

views = sum(
    values["views"]
    for values in aggregator.counts.values()
)

carts = sum(
    values["carts"]
    for values in aggregator.counts.values()
)

purchases = sum(
    values["purchases"]
    for values in aggregator.counts.values()
)

print("Vistas:", views)
print("Carritos:", carts)
print("Compras:", purchases)

assert views == 97130
assert carts == 1215
assert purchases == 1655
assert views + carts + purchases == 100000

print("\n¡Agregación incremental consistente!")


=== ESTADO DEL AGREGADOR ===
Productos agregados: 20621
Eventos contabilizados: 100000
Vistas: 97130
Carritos: 1215
Compras: 1655

¡Agregación incremental consistente!


## 11. Agregación incremental usuario-producto

In [14]:

import importlib
import src.streaming_aggregates as streaming_aggregates

# Recargar el módulo desde el archivo guardado
importlib.reload(streaming_aggregates)

# Verificar las clases disponibles
print("Clases disponibles:")

for name in dir(streaming_aggregates):
    if "Aggregator" in name:
        print("-", name)

# Importar la nueva clase
from src.streaming_aggregates import (
    UserProductInteractionAggregator
)

print("\n¡Importación exitosa!")


Clases disponibles:
- ProductInteractionAggregator
- UserProductInteractionAggregator

¡Importación exitosa!


In [15]:

from src.streaming_aggregates import (
    UserProductInteractionAggregator
)
from src.streaming_ingestion import StreamingIngestion

N = 100_000

ingestion_csr = StreamingIngestion(
    batch_size=1000
)

user_product_agg = UserProductInteractionAggregator()

user_product_agg.process(
    ingestion_csr.process(
        iter_rees46(
            DATA_PATH,
            N,
            chunksize=2000
        )
    )
)

actual_pairs = (
    user_product_agg.to_dataframe()
    .sort_values(["user_id", "product_id"])
    .reset_index(drop=True)
)

# Referencia independiente con pandas
expected_pairs = (
    df.head(N)
    .groupby(
        ["user_id", "product_id", "event_type"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["view", "cart", "purchase"],
        fill_value=0
    )
    .rename(
        columns={
            "view": "views",
            "cart": "carts",
            "purchase": "purchases"
        }
    )
    .reset_index()
    .sort_values(["user_id", "product_id"])
    .reset_index(drop=True)
)

# Normalizar metadatos de pandas
actual_pairs.columns.name = None
expected_pairs.columns.name = None

pd.testing.assert_frame_equal(
    actual_pairs,
    expected_pairs,
    check_dtype=False
)

print("=== AGREGACIÓN USUARIO-PRODUCTO ===")
print("Eventos:", user_product_agg.total_events)
print("Pares únicos:", len(actual_pairs))
print("Usuarios únicos:", actual_pairs["user_id"].nunique())
print("Productos únicos:", actual_pairs["product_id"].nunique())

print("\n¡Agregados equivalentes a pandas!")


=== AGREGACIÓN USUARIO-PRODUCTO ===
Eventos: 100000
Pares únicos: 65919
Usuarios únicos: 20384
Productos únicos: 20621

¡Agregados equivalentes a pandas!


## 12. Construcción de matriz CSR incremental

In [16]:

import numpy as np
from scipy.sparse import csr_matrix

# Utilizamos los agregados ya validados
counts = user_product_agg.counts

# Crear índices únicos y ordenados
user_ids = sorted({user for user, product in counts})
product_ids = sorted({product for user, product in counts})

user_to_idx = {
    user_id: i
    for i, user_id in enumerate(user_ids)
}

product_to_idx = {
    product_id: j
    for j, product_id in enumerate(product_ids)
}

rows = []
cols = []
values = []

for (user_id, product_id), interaction in counts.items():
    rows.append(user_to_idx[user_id])
    cols.append(product_to_idx[product_id])

    # Por ahora, peso = cantidad total de interacciones
    values.append(
        interaction["views"]
        + interaction["carts"]
        + interaction["purchases"]
    )

csr_incremental = csr_matrix(
    (
        np.array(values, dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(user_ids), len(product_ids))
)

print("=== MATRIZ CSR INCREMENTAL ===")
print("Dimensiones:", csr_incremental.shape)
print("Valores no nulos:", csr_incremental.nnz)
print("Suma de interacciones:", csr_incremental.sum())
print(
    "Memoria CSR:",
    f"{(csr_incremental.data.nbytes + csr_incremental.indices.nbytes + csr_incremental.indptr.nbytes) / 1024**2:.3f} MiB"
)

assert csr_incremental.shape == (20384, 20621)
assert csr_incremental.nnz == 65919
assert csr_incremental.sum() == 100000

print("\n¡CSR incremental construida correctamente!")


=== MATRIZ CSR INCREMENTAL ===
Dimensiones: (20384, 20621)
Valores no nulos: 65919
Suma de interacciones: 100000
Memoria CSR: 0.581 MiB

¡CSR incremental construida correctamente!


## 13. Construcción y validación de CSR ponderada

In [17]:

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from src.sparse_matrix import EVENT_WEIGHTS

# Tomamos los agregados usuario-producto ya validados
counts = user_product_agg.counts

rows = []
cols = []
weighted_values = []

for (user_id, product_id), interaction in counts.items():

    row = user_to_idx[user_id]
    col = product_to_idx[product_id]

    weight = (
        interaction["views"] * EVENT_WEIGHTS["view"]
        + interaction["carts"] * EVENT_WEIGHTS["cart"]
        + interaction["purchases"] * EVENT_WEIGHTS["purchase"]
    )

    rows.append(row)
    cols.append(col)
    weighted_values.append(weight)

csr_weighted = csr_matrix(
    (
        np.asarray(weighted_values, dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(user_ids), len(product_ids))
)

print("=== CSR PONDERADA INCREMENTAL ===")
print("Dimensiones:", csr_weighted.shape)
print("Elementos no nulos:", csr_weighted.nnz)
print("Suma ponderada:", int(csr_weighted.sum()))

assert csr_weighted.shape == (20384, 20621)
assert csr_weighted.nnz == 65919

print("\n¡Matriz CSR ponderada construida!")


=== CSR PONDERADA INCREMENTAL ===
Dimensiones: (20384, 20621)
Elementos no nulos: 65919
Suma ponderada: 109050

¡Matriz CSR ponderada construida!


In [18]:

# Referencia independiente de pandas
df_reference = df.head(100_000).copy()

df_reference["weight"] = (
    df_reference["event_type"].map(EVENT_WEIGHTS)
)

reference_weights = (
    df_reference
    .groupby(["user_id", "product_id"])["weight"]
    .sum()
)

# Convertir los agregados de referencia a coordenadas CSR
ref_rows = [
    user_to_idx[user]
    for user, product in reference_weights.index
]

ref_cols = [
    product_to_idx[product]
    for user, product in reference_weights.index
]

csr_reference = csr_matrix(
    (
        reference_weights.to_numpy(dtype=np.int32),
        (ref_rows, ref_cols)
    ),
    shape=csr_weighted.shape
)

# Comparar todas las posiciones, no solo las sumas
difference = csr_weighted - csr_reference

print("=== VALIDACIÓN CSR PONDERADA ===")
print("Forma idéntica:", csr_weighted.shape == csr_reference.shape)
print("NNZ incremental:", csr_weighted.nnz)
print("NNZ referencia:", csr_reference.nnz)
print("Suma incremental:", int(csr_weighted.sum()))
print("Suma referencia:", int(csr_reference.sum()))
print("Diferencias no nulas:", difference.nnz)

assert difference.nnz == 0
assert int(csr_weighted.sum()) == 109050

print("\n¡CSR incremental ponderada equivalente a pandas!")


=== VALIDACIÓN CSR PONDERADA ===
Forma idéntica: True
NNZ incremental: 65919
NNZ referencia: 65919
Suma incremental: 109050
Suma referencia: 109050
Diferencias no nulas: 0

¡CSR incremental ponderada equivalente a pandas!


In [19]:

import importlib
import src.streaming_aggregates as streaming_module

importlib.reload(streaming_module)

# Reconstruir el acumulador con la versión actualizada
agg_csr = streaming_module.UserProductInteractionAggregator()

for (user_id, product_id), counts in user_product_agg.counts.items():
    agg_csr.counts[(user_id, product_id)] = counts.copy()

agg_csr.total_events = user_product_agg.total_events

matrix, users, products = agg_csr.to_csr()

print("=== CSR DESDE EL MÓDULO ===")
print("Dimensiones:", matrix.shape)
print("NNZ:", matrix.nnz)
print("Suma ponderada:", int(matrix.sum()))

# Comparar con la CSR validada anteriormente
assert users == user_ids
assert products == product_ids
assert (matrix != csr_weighted).nnz == 0

print("\n¡Método to_csr validado!")


=== CSR DESDE EL MÓDULO ===
Dimensiones: (20384, 20621)
NNZ: 65919
Suma ponderada: 109050

¡Método to_csr validado!


In [20]:

import inspect
from src.sparse_matrix import UserProductMatrix

print("=== INTERFAZ DE UserProductMatrix ===")

for name in ["__init__", "add_event", "to_csr"]:
    method = getattr(UserProductMatrix, name, None)

    if method is not None:
        print(f"{name}: {inspect.signature(method)}")

print("\n=== MÉTODOS PÚBLICOS ===")
print([
    name for name in dir(UserProductMatrix)
    if not name.startswith("_")
])


=== INTERFAZ DE UserProductMatrix ===
__init__: (self)
add_event: (self, event)
to_csr: (self)

=== MÉTODOS PÚBLICOS ===
['add_event', 'to_csr']


## 14. Comparación directa: CSR original vs. incremental

In [21]:

from src.sparse_matrix import UserProductMatrix
from src.streaming_aggregates import UserProductInteractionAggregator

# 1. Construir CSR con la implementación original
original_csr_builder = UserProductMatrix()

for event in df.head(100_000).to_dict("records"):
    original_csr_builder.add_event(event)

original_result = original_csr_builder.to_csr()

# 2. Construir CSR con el acumulador incremental
incremental_matrix, incremental_users, incremental_products = (
    agg_csr.to_csr()
)

# 3. Comparar las interacciones directamente
# Ambos acumuladores conservan valores por (user_id, product_id).
from src.sparse_matrix import EVENT_WEIGHTS

expected_weights = original_csr_builder.interactions

actual_weights = {
    key: (
        counts["views"] * EVENT_WEIGHTS["view"]
        + counts["carts"] * EVENT_WEIGHTS["cart"]
        + counts["purchases"] * EVENT_WEIGHTS["purchase"]
    )
    for key, counts in agg_csr.counts.items()
}

print("=== COMPARACIÓN ORIGINAL VS STREAMING ===")
print("Pares originales:", len(expected_weights))
print("Pares streaming:", len(actual_weights))
print("Diccionarios idénticos:", expected_weights == actual_weights)

assert expected_weights == actual_weights

print("\n¡Interacciones ponderadas equivalentes!")


=== COMPARACIÓN ORIGINAL VS STREAMING ===
Pares originales: 65919
Pares streaming: 65919
Diccionarios idénticos: True

¡Interacciones ponderadas equivalentes!


In [22]:

import importlib
import src.streaming_aggregates as streaming_module

importlib.reload(streaming_module)

StreamingProductCatalog = (
    streaming_module.StreamingProductCatalog
)

catalog = StreamingProductCatalog()

events_test = [
    {
        "event_id": 1,
        "product_id": 100,
        "event_timestamp": "2026-10-01T10:00:00Z",
        "price": 150.0
    },
    {
        "event_id": 2,
        "product_id": 100,
        "event_timestamp": "2026-10-01T09:00:00Z",
        "price": 200.0
    },
    {
        "event_id": 3,
        "product_id": 100,
        "event_timestamp": "2026-10-01T10:00:00Z",
        "price": 175.0
    }
]

catalog.process(events_test)

product = catalog.get(100)

print("=== CATÁLOGO INCREMENTAL ===")
print("Productos:", len(catalog))
print("Eventos procesados:", catalog.total_events)
print("Precio vigente:", product["price"])
print("Evento seleccionado:", product["event_id"])

assert len(catalog) == 1
assert product["event_id"] == 3
assert product["price"] == 175.0

print("\n¡Selección temporal validada!")


=== CATÁLOGO INCREMENTAL ===
Productos: 1
Eventos procesados: 3
Precio vigente: 175.0
Evento seleccionado: 3

¡Selección temporal validada!


## 15. Catálogo incremental de productos REES46

In [23]:

from src.streaming_ingestion import StreamingIngestion
from src.streaming_aggregates import StreamingProductCatalog

N = 100_000

ingestion_catalog = StreamingIngestion(batch_size=1000)
catalog_streaming = StreamingProductCatalog()

catalog_streaming.process(
    ingestion_catalog.process(
        iter_rees46(
            DATA_PATH,
            N,
            chunksize=2000
        )
    )
)

print("=== CATÁLOGO INCREMENTAL REES46 ===")
print("Eventos procesados:", catalog_streaming.total_events)
print("Productos únicos:", len(catalog_streaming))
print("Eventos duplicados:", ingestion_catalog.metrics["duplicates"])

assert catalog_streaming.total_events == 100_000
assert len(catalog_streaming) == 20_621

print("\n¡Catálogo incremental construido correctamente!")


=== CATÁLOGO INCREMENTAL REES46 ===
Eventos procesados: 100000
Productos únicos: 20621
Eventos duplicados: 0

¡Catálogo incremental construido correctamente!


In [24]:

import pandas as pd

# Referencia basada en los datos originales
reference_catalog = (
    df.head(100_000)
    .sort_values(
        ["event_timestamp", "event_id"]
    )
    .drop_duplicates(
        subset=["product_id"],
        keep="last"
    )
    .set_index("product_id")
)

differences = []

for product_id, event in catalog_streaming.products.items():

    expected = reference_catalog.loc[product_id]

    if (
        event["event_id"] != expected["event_id"]
        or event["price"] != expected["price"]
    ):
        differences.append(product_id)

print("=== VALIDACIÓN TEMPORAL DEL CATÁLOGO ===")
print("Productos originales:", len(reference_catalog))
print("Productos streaming:", len(catalog_streaming))
print("Productos con diferencias:", len(differences))

assert len(differences) == 0
assert len(reference_catalog) == len(catalog_streaming)

print("\n¡Catálogo incremental equivalente a pandas!")


=== VALIDACIÓN TEMPORAL DEL CATÁLOGO ===
Productos originales: 20621
Productos streaming: 20621
Productos con diferencias: 0

¡Catálogo incremental equivalente a pandas!


In [25]:

import pandas as pd

# Construir referencia independiente con pandas
reference_catalog = (
    df.head(100_000)
    .sort_values(
        ["event_timestamp", "event_id"]
    )
    .drop_duplicates(
        subset=["product_id"],
        keep="last"
    )
    .set_index("product_id")
)

differences = []

for product_id, event in catalog_streaming.products.items():

    expected = reference_catalog.loc[product_id]

    if (
        event["event_id"] != expected["event_id"]
        or event["price"] != expected["price"]
    ):
        differences.append(product_id)

print("=== VALIDACIÓN TEMPORAL DEL CATÁLOGO ===")
print("Productos de referencia:", len(reference_catalog))
print("Productos streaming:", len(catalog_streaming))
print("Productos con diferencias:", len(differences))

assert len(reference_catalog) == len(catalog_streaming)
assert len(differences) == 0

print("\n¡Catálogo incremental equivalente a pandas!")


=== VALIDACIÓN TEMPORAL DEL CATÁLOGO ===
Productos de referencia: 20621
Productos streaming: 20621
Productos con diferencias: 0

¡Catálogo incremental equivalente a pandas!


## 16. Comparación del catálogo incremental con ProductStore

In [26]:

import inspect
from src.storage import ProductStore

print("=== INTERFAZ DE ProductStore ===")

print("Constructor:", inspect.signature(ProductStore))

print("\n=== MÉTODOS PÚBLICOS ===")

for name in dir(ProductStore):
    if not name.startswith("_"):
        method = getattr(ProductStore, name)

        if callable(method):
            try:
                print(f"{name}: {inspect.signature(method)}")
            except (TypeError, ValueError):
                print(name)
        else:
            print(f"{name}: atributo")

print("\n=== CÓDIGO DE ProductStore ===")
print(inspect.getsource(ProductStore))


=== INTERFAZ DE ProductStore ===
Constructor: ()

=== MÉTODOS PÚBLICOS ===
add_product: (self, product, timestamp, event_id)
get_product: (self, product_id)

=== CÓDIGO DE ProductStore ===
class ProductStore:
    """
    Catálogo de productos mediante Hash Table.

    Conserva los atributos del evento más reciente
    según timestamp y event_id.

    Lookup: O(1) promedio.
    """

    def __init__(self):
        self.products = {}
        self.last_update = {}

    def add_product(self, product, timestamp, event_id):
        product_id = product["product_id"]

        # Timestamp y event_id sirven para determinar
        # qué registro debe prevalecer.
        version = (timestamp, int(event_id))

        if (
            product_id not in self.last_update
            or version > self.last_update[product_id]
        ):
            self.products[product_id] = product.copy()
            self.last_update[product_id] = version
            return True

        return False

    def get_pr

## 17. Comparación directa con ProductStore

In [27]:

from src.storage import ProductStore
import pandas as pd

store_original = ProductStore()

for event in df.head(100_000).to_dict("records"):
    store_original.add_product(
        product=event,
        timestamp=event["event_timestamp"],
        event_id=event["event_id"]
    )

print("=== COMPARACIÓN DE CATÁLOGOS ===")
print("Productos originales:", len(store_original))
print("Productos streaming:", len(catalog_streaming))

assert len(store_original) == len(catalog_streaming) == 20621

# Inspeccionar un ejemplo de ambas estructuras
product_id = next(iter(catalog_streaming.products))

print("\n=== EJEMPLO DE PRODUCTO ===")
print("Product ID:", product_id)
print("ProductStore:", store_original.get_product(product_id))
print("Streaming:", catalog_streaming.get(product_id))
print("Última actualización:", store_original.last_update.get(product_id))


=== COMPARACIÓN DE CATÁLOGOS ===
Productos originales: 20621
Productos streaming: 20621

=== EJEMPLO DE PRODUCTO ===
Product ID: 44600062
ProductStore: {'event_time': '2019-10-01 00:00:15 UTC', 'event_type': 'view', 'product_id': 44600062, 'category_id': 2103807459595387724, 'category_code': nan, 'brand': 'shiseido', 'price': 35.79, 'user_id': 541312140, 'user_session': '72d76fde-8bb3-4e00-8c23-a032dfed738c', 'event_id': 13, 'event_timestamp': Timestamp('2019-10-01 00:00:15+0000', tz='UTC')}
Streaming: {'event_time': '2019-10-01 00:00:15 UTC', 'event_type': 'view', 'product_id': 44600062, 'category_id': 2103807459595387724, 'category_code': nan, 'brand': 'shiseido', 'price': 35.79, 'user_id': 541312140, 'user_session': '72d76fde-8bb3-4e00-8c23-a032dfed738c', 'event_id': 13, 'event_timestamp': Timestamp('2019-10-01 00:00:15+0000', tz='UTC')}
Última actualización: (Timestamp('2019-10-01 00:00:15+0000', tz='UTC'), 13)


In [28]:

import inspect
from src.storage import ProductStore

print("=== REGLAS DE ACTUALIZACIÓN ===")
print(inspect.getsource(ProductStore.add_product))

print("\n=== RECUPERACIÓN DE PRODUCTOS ===")
print(inspect.getsource(ProductStore.get_product))


=== REGLAS DE ACTUALIZACIÓN ===
    def add_product(self, product, timestamp, event_id):
        product_id = product["product_id"]

        # Timestamp y event_id sirven para determinar
        # qué registro debe prevalecer.
        version = (timestamp, int(event_id))

        if (
            product_id not in self.last_update
            or version > self.last_update[product_id]
        ):
            self.products[product_id] = product.copy()
            self.last_update[product_id] = version
            return True

        return False


=== RECUPERACIÓN DE PRODUCTOS ===
    def get_product(self, product_id):
        return self.products.get(product_id)



## 18. Equivalencia completa de ProductStore y StreamingProductCatalog

In [29]:

import pandas as pd

original_products = store_original.products
streaming_products = catalog_streaming.products

# 1. Comparar identificadores de productos
original_ids = set(original_products)
streaming_ids = set(streaming_products)

assert original_ids == streaming_ids

# 2. Comparar todos los atributos de cada producto
differences = []

def same_value(a, b):
    # Considerar dos valores faltantes como equivalentes
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b

for product_id in original_ids:
    original_event = original_products[product_id]
    streaming_event = streaming_products[product_id]

    if original_event.keys() != streaming_event.keys():
        differences.append((product_id, "columnas"))
        continue

    for field in original_event:
        if not same_value(
            original_event[field],
            streaming_event[field]
        ):
            differences.append((product_id, field))
            break

print("=== EQUIVALENCIA COMPLETA DE CATÁLOGOS ===")
print("Productos originales:", len(original_ids))
print("Productos streaming:", len(streaming_ids))
print("Productos con diferencias:", len(differences))

if differences:
    print("Primeras diferencias:", differences[:10])

assert len(differences) == 0

print("\n¡Catálogos equivalentes campo por campo!")


=== EQUIVALENCIA COMPLETA DE CATÁLOGOS ===
Productos originales: 20621
Productos streaming: 20621
Productos con diferencias: 0

¡Catálogos equivalentes campo por campo!


## 19. Construcción de B+Tree desde catálogo incremental

In [30]:

from src.bplus_tree import BPlusTree
import time

# Índice construido desde el catálogo incremental
bpt_streaming = BPlusTree(max_keys=32)

start = time.perf_counter()

for product_id, event in catalog_streaming.products.items():
    price_cents = int(round(float(event["price"]) * 100))

    bpt_streaming.insert(
        price_cents,
        product_id
    )

elapsed = time.perf_counter() - start

print("=== B+TREE DESDE CATÁLOGO INCREMENTAL ===")
print("Productos en catálogo:", len(catalog_streaming))
print("Entradas en B+Tree:", len(bpt_streaming))
print(f"Tiempo de construcción: {elapsed:.4f} s")

assert len(bpt_streaming) == 20621

print("\n¡B+Tree incremental construido!")


=== B+TREE DESDE CATÁLOGO INCREMENTAL ===
Productos en catálogo: 20621
Entradas en B+Tree: 20621
Tiempo de construcción: 0.0443 s

¡B+Tree incremental construido!


In [31]:

import inspect

print("=== INTERFAZ DEL B+TREE ===")
print("insert:", inspect.signature(BPlusTree.insert))
print("range_search:", inspect.signature(BPlusTree.range_search))

# Consultar productos de S/100 a S/300
results_range = bpt_streaming.range_search(
    10000,
    30000
)

print("\n=== CONSULTA POR RANGO ===")
print("Rango: S/100 a S/300")
print("Tipo de resultado:", type(results_range).__name__)
print("Cantidad de resultados:", len(results_range))
print("Primeros resultados:", results_range[:5])


=== INTERFAZ DEL B+TREE ===
insert: (self, key, value)
range_search: (self, min_key, max_key)

=== CONSULTA POR RANGO ===
Rango: S/100 a S/300
Tipo de resultado: list
Cantidad de resultados: 5909
Primeros resultados: [4501638, 12719990, 1005007, 1005103, 31501009]


## 20. Validación de B+Tree original vs. incremental

In [32]:

from src.bplus_tree import BPlusTree

# Construir otro índice usando ProductStore original
bpt_original = BPlusTree(max_keys=32)

for product_id, event in store_original.products.items():
    price_cents = int(round(float(event["price"]) * 100))
    bpt_original.insert(price_cents, product_id)

# Probar diferentes rangos de precios (en céntimos)
ranges = [
    (0, 5000),        # S/0 a S/50
    (5000, 10000),    # S/50 a S/100
    (10000, 30000),   # S/100 a S/300
    (30000, 100000),  # S/300 a S/1,000
    (0, 10000000),    # S/0 a S/100,000
]

print("=== COMPARACIÓN DE B+TREE ===")

for min_price, max_price in ranges:
    original_results = bpt_original.range_search(
        min_price, max_price
    )

    streaming_results = bpt_streaming.range_search(
        min_price, max_price
    )

    # Comparar conjuntos de productos
    original_set = set(original_results)
    streaming_set = set(streaming_results)

    identical = original_set == streaming_set

    print(
        f"S/{min_price/100:.2f} - S/{max_price/100:.2f}: "
        f"Original={len(original_results):,}, "
        f"Streaming={len(streaming_results):,}, "
        f"Coinciden={identical}"
    )

    assert len(original_results) == len(streaming_results)
    assert original_set == streaming_set

print("\n¡B+Tree incremental equivalente al original!")


=== COMPARACIÓN DE B+TREE ===
S/0.00 - S/50.00: Original=6,061, Streaming=6,061, Coinciden=True
S/50.00 - S/100.00: Original=4,473, Streaming=4,473, Coinciden=True
S/100.00 - S/300.00: Original=5,909, Streaming=5,909, Coinciden=True
S/300.00 - S/1000.00: Original=3,360, Streaming=3,360, Coinciden=True
S/0.00 - S/100000.00: Original=20,621, Streaming=20,621, Coinciden=True

¡B+Tree incremental equivalente al original!


In [33]:

print("=== B+TREE VS BÚSQUEDA LINEAL ===")

for min_price, max_price in ranges:

    # Referencia directa del catálogo
    linear_results = {
        product_id
        for product_id, event in catalog_streaming.products.items()
        if min_price <= int(round(float(event["price"]) * 100)) <= max_price
    }

    tree_results = set(
        bpt_streaming.range_search(
            min_price, max_price
        )
    )

    assert tree_results == linear_results

    print(
        f"Rango S/{min_price/100:.2f} - S/{max_price/100:.2f}: "
        f"{len(tree_results):,} productos correctos"
    )

print("\n¡B+Tree validado contra búsqueda lineal!")


=== B+TREE VS BÚSQUEDA LINEAL ===
Rango S/0.00 - S/50.00: 6,061 productos correctos
Rango S/50.00 - S/100.00: 4,473 productos correctos
Rango S/100.00 - S/300.00: 5,909 productos correctos
Rango S/300.00 - S/1000.00: 3,360 productos correctos
Rango S/0.00 - S/100000.00: 20,621 productos correctos

¡B+Tree validado contra búsqueda lineal!


In [34]:

import inspect
from src.product_features import build_product_features

print("=== IMPLEMENTACIÓN ORIGINAL DE CARACTERÍSTICAS ===")
print(inspect.signature(build_product_features))
print(inspect.getsource(build_product_features))


=== IMPLEMENTACIÓN ORIGINAL DE CARACTERÍSTICAS ===
(events)
def build_product_features(events):
    """
    Genera características de productos a partir
    de eventos REES46 procesados.

    Precio: mediana observada por producto.
    Comportamiento: frecuencia por tipo de evento.
    Transformación: log1p.
    """

    df = pd.DataFrame(events)

    if df.empty:
        raise ValueError("No hay eventos para procesar")

    required = {
        "product_id", "event_type", "price"
    }

    if not required.issubset(df.columns):
        raise ValueError("Faltan columnas requeridas")

    # Contar interacciones por producto y tipo
    counts = pd.crosstab(
        df["product_id"],
        df["event_type"]
    )

    for event_type in ["view", "cart", "purchase"]:
        if event_type not in counts.columns:
            counts[event_type] = 0

    # Calcular precio representativo
    prices = df.groupby("product_id")["price"].median()

    features = pd.DataFrame({
        "product_id":

In [35]:

import inspect
from src.product_features import build_product_features

# Mostrar el código completo en un formato más legible
source = inspect.getsource(build_product_features)

for number, line in enumerate(source.splitlines(), start=1):
    print(f"{number:03d} | {line}")


001 | def build_product_features(events):
002 |     """
003 |     Genera características de productos a partir
004 |     de eventos REES46 procesados.
005 | 
006 |     Precio: mediana observada por producto.
007 |     Comportamiento: frecuencia por tipo de evento.
008 |     Transformación: log1p.
009 |     """
010 | 
011 |     df = pd.DataFrame(events)
012 | 
013 |     if df.empty:
014 |         raise ValueError("No hay eventos para procesar")
015 | 
016 |     required = {
017 |         "product_id", "event_type", "price"
018 |     }
019 | 
020 |     if not required.issubset(df.columns):
021 |         raise ValueError("Faltan columnas requeridas")
022 | 
023 |     # Contar interacciones por producto y tipo
024 |     counts = pd.crosstab(
025 |         df["product_id"],
026 |         df["event_type"]
027 |     )
028 | 
029 |     for event_type in ["view", "cart", "purchase"]:
030 |         if event_type not in counts.columns:
031 |             counts[event_type] = 0
032 | 
033 |     # C

In [36]:

prices_per_product = (
    df.head(100_000)
    .groupby("product_id")["price"]
    .count()
)

print("=== PRECIOS POR PRODUCTO ===")
print("Productos:", len(prices_per_product))
print("Mediana de registros:", prices_per_product.median())
print("Promedio de registros:", round(prices_per_product.mean(), 2))
print("Máximo de registros:", prices_per_product.max())

print("\nDistribución:")
print(
    prices_per_product.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)


=== PRECIOS POR PRODUCTO ===
Productos: 20621
Mediana de registros: 2.0
Promedio de registros: 4.85
Máximo de registros: 1130

Distribución:
count    20621.000000
mean         4.849425
std         19.970387
min          1.000000
50%          2.000000
90%          8.000000
95%         15.000000
99%         49.800000
max       1130.000000
Name: price, dtype: float64


## 21. Cálculo incremental de medianas de precios

In [37]:

import importlib
import pandas as pd
import numpy as np

import src.streaming_aggregates as streaming_module
from src.streaming_ingestion import StreamingIngestion

importlib.reload(streaming_module)

StreamingProductPriceMedian = (
    streaming_module.StreamingProductPriceMedian
)

N = 100_000

ingestion_prices = StreamingIngestion(batch_size=1000)
price_aggregator = StreamingProductPriceMedian()

price_aggregator.process(
    ingestion_prices.process(
        iter_rees46(
            DATA_PATH,
            N,
            chunksize=2000
        )
    )
)

# Medianas generadas por streaming
actual_medians = (
    price_aggregator.to_dataframe()
    .sort_values("product_id")
    .reset_index(drop=True)
)

# Referencia independiente con pandas
expected_medians = (
    df.head(N)
    .groupby("product_id")["price"]
    .median()
    .reset_index(name="median_price")
    .sort_values("product_id")
    .reset_index(drop=True)
)

# Comparación numérica con tolerancia mínima
pd.testing.assert_frame_equal(
    actual_medians,
    expected_medians,
    check_dtype=False,
    rtol=1e-12,
    atol=1e-12
)

print("=== MEDIANAS INCREMENTALES ===")
print("Precios acumulados:", price_aggregator.total_prices)
print("Productos:", len(actual_medians))

max_difference = np.max(
    np.abs(
        actual_medians["median_price"].to_numpy()
        - expected_medians["median_price"].to_numpy()
    )
)

print("Diferencia máxima:", max_difference)

assert len(actual_medians) == 20621
assert price_aggregator.total_prices == 100000

print("\n¡Medianas incrementales equivalentes a pandas!")


=== MEDIANAS INCREMENTALES ===
Precios acumulados: 100000
Productos: 20621
Diferencia máxima: 0.0

¡Medianas incrementales equivalentes a pandas!


## 22. Características incrementales para KD-Tree

In [38]:

import numpy as np
import pandas as pd

# Contadores por producto validados anteriormente
df_counts = aggregator.to_dataframe()

# Medianas de precios validadas
df_prices = price_aggregator.to_dataframe()

# Combinar ambas estructuras
features_streaming = df_counts.merge(
    df_prices,
    on="product_id",
    how="inner",
    validate="one_to_one"
)

# Aplicar las mismas transformaciones log1p
features_streaming["log_price"] = np.log1p(
    features_streaming["median_price"]
)
features_streaming["log_views"] = np.log1p(
    features_streaming["views"]
)
features_streaming["log_carts"] = np.log1p(
    features_streaming["carts"]
)
features_streaming["log_purchases"] = np.log1p(
    features_streaming["purchases"]
)

feature_columns = [
    "log_price",
    "log_views",
    "log_carts",
    "log_purchases"
]

features_streaming = (
    features_streaming[
        ["product_id"] + feature_columns
    ]
    .sort_values("product_id")
    .reset_index(drop=True)
)

print("=== CARACTERÍSTICAS INCREMENTALES ===")
print("Productos:", len(features_streaming))
print("Columnas:", features_streaming.columns.tolist())

display(features_streaming.head())

assert len(features_streaming) == 20621

assert np.isfinite(
    features_streaming[feature_columns].to_numpy()
).all()

print("\n¡Características incrementales construidas!")


=== CARACTERÍSTICAS INCREMENTALES ===
Productos: 20621
Columnas: ['product_id', 'log_price', 'log_views', 'log_carts', 'log_purchases']


,product_id,log_price,log_views,log_carts,log_purchases
0,1001588,4.862213,2.197225,0.0,0.0
1,1002062,4.358502,0.693147,0.0,0.0
2,1002098,6.016206,2.302585,0.0,0.0
3,1002099,5.917307,4.330733,0.0,0.0
4,1002100,6.016206,2.639057,0.0,0.0



¡Características incrementales construidas!


In [39]:

from src.product_features import build_product_features

# Implementación original
features_original = build_product_features(
    df.head(100_000).to_dict("records")
)

print("=== ESTRUCTURA ORIGINAL ===")
print("Columnas:", features_original.columns.tolist())
print("Dimensiones:", features_original.shape)

# Comparar por product_id, independientemente del orden
assert "product_id" in features_original.columns

original_sorted = (
    features_original[
        ["product_id"] + feature_columns
    ]
    .sort_values("product_id")
    .reset_index(drop=True)
)

streaming_sorted = features_streaming.copy()

assert original_sorted["product_id"].equals(
    streaming_sorted["product_id"]
)

# Comparación numérica de las cuatro características
differences = np.abs(
    original_sorted[feature_columns].to_numpy()
    - streaming_sorted[feature_columns].to_numpy()
)

max_difference = float(differences.max())

print("\n=== COMPARACIÓN DE CARACTERÍSTICAS ===")
print("Productos originales:", len(original_sorted))
print("Productos streaming:", len(streaming_sorted))
print("Diferencia máxima:", max_difference)

np.testing.assert_allclose(
    original_sorted[feature_columns].to_numpy(),
    streaming_sorted[feature_columns].to_numpy(),
    rtol=1e-12,
    atol=1e-12
)

print("\n¡Características equivalentes al método original!")


=== ESTRUCTURA ORIGINAL ===
Columnas: ['product_id', 'price', 'views', 'carts', 'purchases', 'log_price', 'log_views', 'log_carts', 'log_purchases']
Dimensiones: (20621, 9)

=== COMPARACIÓN DE CARACTERÍSTICAS ===
Productos originales: 20621
Productos streaming: 20621
Diferencia máxima: 0.0

¡Características equivalentes al método original!


## 23. Construcción del KD-Tree incremental

In [40]:

import inspect
import time
import numpy as np

from src.multidimensional_index import ProductKDTree

print("=== INTERFAZ KD-TREE ===")
print("Constructor:", inspect.signature(ProductKDTree))
print("fit:", inspect.signature(ProductKDTree.fit))
print("query:", inspect.signature(ProductKDTree.query))

# Mantener exactamente el orden de características
feature_columns = [
    "log_price",
    "log_views",
    "log_carts",
    "log_purchases"
]

product_ids = features_streaming["product_id"].to_numpy()
X_streaming = features_streaming[
    feature_columns
].to_numpy(dtype=np.float64)

assert X_streaming.shape == (20621, 4)
assert np.isfinite(X_streaming).all()

# Construir el índice
kdtree_streaming = ProductKDTree(leaf_size=40)

start = time.perf_counter()

kdtree_streaming.fit(
    product_ids,
    X_streaming
)

elapsed = time.perf_counter() - start

print("\n=== KD-TREE INCREMENTAL ===")
print("Productos indexados:", len(product_ids))
print("Dimensiones:", X_streaming.shape)
print(f"Tiempo de construcción: {elapsed:.4f} s")

print("\n¡KD-Tree incremental construido!")


=== INTERFAZ KD-TREE ===
Constructor: (leaf_size=40)
fit: (self, product_ids, features)
query: (self, product_id, k=5, method='kd_tree')

=== KD-TREE INCREMENTAL ===
Productos indexados: 20621
Dimensiones: (20621, 4)
Tiempo de construcción: 0.0235 s

¡KD-Tree incremental construido!


In [41]:

# Utilizar la referencia original ya validada
X_original = original_sorted[
    feature_columns
].to_numpy(dtype=np.float64)

ids_original = original_sorted[
    "product_id"
].to_numpy()

# Comprobación previa indispensable
assert np.array_equal(product_ids, ids_original)

np.testing.assert_array_equal(
    X_streaming,
    X_original
)

# KD-Tree de referencia
kdtree_original = ProductKDTree(leaf_size=40)

kdtree_original.fit(
    ids_original,
    X_original
)

print("=== COMPARACIÓN DE KD-TREE ===")

# Consultar 20 productos distribuidos por el catálogo
sample_ids = product_ids[
    np.linspace(
        0,
        len(product_ids) - 1,
        20,
        dtype=int
    )
]

for product_id in sample_ids:
    original_result = kdtree_original.query(
        product_id,
        k=5,
        method="kd_tree"
    )

    streaming_result = kdtree_streaming.query(
        product_id,
        k=5,
        method="kd_tree"
    )

    assert original_result == streaming_result

print("Consultas evaluadas:", len(sample_ids))
print("Resultados idénticos:", len(sample_ids))

print("\n¡KD-Tree incremental equivalente al original!")


=== COMPARACIÓN DE KD-TREE ===
Consultas evaluadas: 20
Resultados idénticos: 20

¡KD-Tree incremental equivalente al original!


## 24. Validación del pipeline incremental integrado

In [42]:

import time

from src.streaming_pipeline import run_streaming_pipeline

N = 100_000

start = time.perf_counter()

streaming_result = run_streaming_pipeline(
    iter_rees46(DATA_PATH, N, chunksize=2000),
    batch_size=1000
)

elapsed = time.perf_counter() - start
metrics = streaming_result["metrics"]

print("=== PIPELINE INCREMENTAL INTEGRADO ===")
print(f"Tiempo total: {elapsed:.4f} s")

for key, value in metrics.items():
    print(f"{key}: {value}")

# Verificar las estructuras principales
assert metrics["received"] == 100_000
assert metrics["output"] == 100_000

assert metrics["catalog_products"] == 20_621
assert metrics["csr_users"] == 20_384
assert metrics["csr_products"] == 20_621
assert metrics["csr_nnz"] == 65_919
assert metrics["csr_weight_sum"] == 109_050
assert metrics["bptree_products"] == 20_621
assert metrics["kdtree_products"] == 20_621

# Comparar las características previamente validadas
streaming_features = (
    streaming_result["features"]
    .set_index("product_id")
    .loc[original_sorted["product_id"]]
)

np.testing.assert_allclose(
    streaming_features[feature_columns].to_numpy(),
    original_sorted[feature_columns].to_numpy(),
    rtol=1e-12,
    atol=1e-12
)

print("\n¡Pipeline incremental integrado validado!")


=== PIPELINE INCREMENTAL INTEGRADO ===
Tiempo total: 1.8072 s
received: 100000
valid: 100000
invalid: 0
duplicates: 0
output: 100000
catalog_products: 20621
csr_users: 20384
csr_products: 20621
csr_nnz: 65919
csr_weight_sum: 109050
bptree_products: 20621
kdtree_products: 20621

¡Pipeline incremental integrado validado!


## 25. Benchmark integral repetido: original vs streaming

In [43]:

import pandas as pd

benchmark_full = pd.DataFrame({
    "metodo": ["original"] * 5 + ["streaming"] * 5,
    "tiempo_s": [
        1.7739, 1.7616, 1.7651, 1.7643, 1.7754,
        1.5705, 1.5730, 1.6475, 1.7878, 1.8913
    ],
    "throughput": [
        56371, 56766, 56654, 56679, 56325,
        63673, 63573, 60699, 55936, 52873
    ],
    "rss_incremento_mib": [
        155.18, 155.86, 155.36, 155.71, 155.39,
        77.87, 76.29, 77.91, 77.87, 78.16
    ]
})

resumen_full = (
    benchmark_full.groupby("metodo")
    .agg(
        tiempo_mediano=("tiempo_s", "median"),
        throughput_mediano=("throughput", "median"),
        rss_mediano=("rss_incremento_mib", "median"),
        tiempo_min=("tiempo_s", "min"),
        tiempo_max=("tiempo_s", "max")
    )
)

original = resumen_full.loc["original"]
streaming = resumen_full.loc["streaming"]

ahorro_memoria = (
    1 - streaming["rss_mediano"] / original["rss_mediano"]
) * 100

mejora_tiempo = (
    1 - streaming["tiempo_mediano"] /
    original["tiempo_mediano"]
) * 100

print("=== BENCHMARK INTEGRAL ===")
display(resumen_full.round(4))
print(f"Ahorro memoria adicional: {ahorro_memoria:.2f}%")
print(f"Reducción tiempo mediano: {mejora_tiempo:.2f}%")

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_full.to_csv(
    results_dir / "benchmark_full_pipeline_raw.csv",
    index=False
)

resumen_full.to_csv(
    results_dir / "benchmark_full_pipeline_summary.csv"
)

print("\n¡Benchmark integral guardado!")


=== BENCHMARK INTEGRAL ===


,tiempo_mediano,throughput_mediano,rss_mediano,tiempo_min,tiempo_max
metodo,,,,,
original,1.7651,56654.0,155.39,1.7616,1.7754
streaming,1.6475,60699.0,77.87,1.5705,1.8913


Ahorro memoria adicional: 49.89%
Reducción tiempo mediano: 6.66%

¡Benchmark integral guardado!
